# <span style="font-width:bold; font-size: 3rem; color:#1EB182;">**Garmin Companion**</span><span style="font-width:bold; font-size: 3rem; color:#333;"> - 04: Feature Monitoring</span>

<span style="font-width:bold; font-size: 1.4rem;">Native drift detection, prediction monitoring, and alerts.</span>

> **Not medical advice.** This is a personal training-readiness & recovery monitoring system, not a diagnosis or injury-prediction tool.


We use Hopsworks **native feature monitoring** (A7) instead of a bespoke monitoring layer: scheduled statistics, threshold comparison, distribution drift (PSI), a training-dataset reference window, and prediction drift on the logging feature group. Crons are demo-tuned; the cleanup loop keeps the notebook idempotent.

In [ ]:
!pip install -U 'hopsworks[python]' --quiet

## <span style='color:#ff5f27'>📝 Connect &amp; clean up prior configs</span>

In [ ]:
import hopsworks

project = hopsworks.login()
fs = project.get_feature_store()

daily_fg = fs.get_feature_group("fg_garmin_daily_summary_raw", 1)
readiness_fv = fs.get_feature_view("fv_readiness_daily", 1)

# Idempotency: delete any monitoring configs from a previous run.
for obj in (daily_fg, readiness_fv):
    try:
        for cfg in obj.get_feature_monitoring_configs():
            cfg.delete()
    except Exception as exc:
        print("cleanup skipped:", exc)

## <span style='color:#ff5f27'>📊 1. Scheduled statistics on a raw feature group</span>

In [ ]:
daily_stats = daily_fg.create_statistics_monitoring(
    name="daily_summary_scheduled_stats",
    description="Daily descriptive statistics over the last week of data",
    cron_expression="0 0 12 ? * * *",
).with_detection_window(time_offset="1w").save()

## <span style='color:#ff5f27'>📈 2. Threshold comparison on resting HR</span>

In [ ]:
rhr_monitor = daily_fg.create_feature_monitoring(
    name="resting_hr_shift",
    feature_name="resting_hr",
    description="Alert if weekly mean resting HR drifts from the prior 4 weeks",
    cron_expression="0 0 12 ? * * *",
).with_detection_window(
    time_offset="1w", window_length="1w",
).with_reference_window(
    time_offset="4w", window_length="3w",
).compare_on(metric="mean", threshold=3.0).save()

## <span style='color:#ff5f27'>🌗 3. Distribution drift (PSI) vs the training dataset</span>

A `TRAINING_DATASET` reference window detects *input drift* away from what the model was trained on — a strong retraining trigger (A7).

In [ ]:
drift_monitor = readiness_fv.create_feature_monitoring(
    name="readiness_input_drift",
    feature_name="d_resting_hr",
    description="PSI of resting HR vs the training dataset",
    cron_expression="0 0 12 ? * * *",
).with_detection_window(
    time_offset="1w",
).with_reference_training_dataset(
    training_dataset_version=1,
).compare_on_distribution(metric="PSI", threshold=0.2).save()

## <span style='color:#ff5f27'>🔔 4. Prediction drift on the logging feature group &amp; alerts</span>

Feature monitoring anomalies, data-validation failures, and job failures all feed the native **Alerts** framework (email / Slack / PagerDuty / webhook). Configure receivers under *Project Settings → Alerts*, then attach an alert to any monitoring config. Add a **freshness** expectation (max `event_time` lag) and align cron to Garmin's sync cadence so partially-arrived data does not raise false drift alarms (A7).

In [ ]:
# The readiness logging FG (populated by fv.log) is itself monitorable for
# prediction drift, e.g. a shift in the readiness_class distribution over time.
for cfg in readiness_fv.get_feature_monitoring_configs():
    print(cfg.name)

✅ The system now has data validation, native feature monitoring, prediction logging, and alerting — a closed observability loop, all on Hopsworks primitives.